# simple test

对于前期实现基本的快速测试，此处：

软件训练完整分类器：784 -> 128 -> 10

FPGA 先只实现第一层：fc1 + relu

导出第一层输入、权重、bias、int32 累加 golden、relu 输出 golden

In [2]:
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
DATA_DIR = "./data"
OUT_DIR = "./route_a_output"
MODEL_PATH = os.path.join(OUT_DIR, "mlp_route_a.pth")

os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 关于模型：模型进行完整训练，但导出时只导出第一层用于 FPGA

In [4]:
class MLPRouteA(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128, bias=True)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10, bias=True)

    def forward(self, x):
        x = x.view(-1, 784)
        h = self.fc1(x)
        h_relu = self.relu(h)
        y = self.fc2(h_relu)
        return y

# 关于数据：训练时仍然可以用标准 Normalize，因为这里只是先训练出一个可用分类器

In [6]:
def get_train_test_loaders(batch_size=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST(
        DATA_DIR, train=True, download=True, transform=transform
    )
    test_dataset = datasets.MNIST(
        DATA_DIR, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# 训练 / 测试

In [7]:
def train_model(num_epochs=5, lr=1e-3):
    train_loader, test_loader = get_train_test_loaders()

    model = MLPRouteA().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for data, target in train_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            logits = model(data)
            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * data.size(0)

        avg_loss = running_loss / len(train_loader.dataset)
        acc = evaluate_model(model, test_loader)
        print(f"[Epoch {epoch+1}/{num_epochs}] loss={avg_loss:.6f}, test_acc={acc:.4f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Model saved to: {MODEL_PATH}")
    return model


def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            logits = model(data)
            pred = logits.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

    return correct / total


#   关于量化函数：FPGA 只验证第一层，采用固定输入量化策略

In [8]:
INT8_QMIN = -128
INT8_QMAX = 127
INT32_QMIN = -2147483648
INT32_QMAX = 2147483647

def clamp(x, low, high):
    return max(low, min(high, x))

def quantize_affine_tensor(tensor_fp, scale, zero_point, qmin=-128, qmax=127, dtype=torch.int8):
    q = torch.round(tensor_fp / scale + zero_point)
    q = torch.clamp(q, qmin, qmax)
    return q.to(dtype)

def calc_symmetric_scale(tensor_fp, num_bits=8):
    qmax = (2 ** (num_bits - 1)) - 1
    max_abs = max(abs(tensor_fp.min().item()), abs(tensor_fp.max().item()))
    if max_abs == 0:
        return 1.0
    return max_abs / qmax

def quantize_weight_symmetric_int8(weight_fp):
    scale = calc_symmetric_scale(weight_fp, num_bits=8)
    zp = 0
    weight_q = quantize_affine_tensor(weight_fp, scale, zp, -128, 127, torch.int8)
    return weight_q, scale, zp

def quantize_bias_int32(bias_fp, input_scale, weight_scale):
    bias_scale = input_scale * weight_scale
    bias_q = torch.round(bias_fp / bias_scale)
    bias_q = torch.clamp(bias_q, INT32_QMIN, INT32_QMAX).to(torch.int32)
    return bias_q, bias_scale

def quantize_input_hw_style(img_fp_01):
    """
    img_fp_01: 原始像素 [0,1]
    统一策略：
        x_u8 = round(x * 255)
        x_q  = x_u8 - 128
    对应：
        input_scale = 1/255
        input_zero_point = -128
    """
    input_scale = 1.0 / 255.0
    input_zero_point = -128

    x_u8 = torch.round(img_fp_01 * 255.0)
    x_u8 = torch.clamp(x_u8, 0, 255)
    x_q = x_u8.to(torch.int16) - 128
    x_q = torch.clamp(x_q, -128, 127).to(torch.int8)

    return x_q, input_scale, input_zero_point

# 第一层整数推理 golden，只对应 fc1 + relu

In [9]:
def fc1_integer_reference(x_q, w_q, b_q, x_zp=-128, w_zp=0):
    """
    x_q: [784] int8
    w_q: [128, 784] int8
    b_q: [128] int32

    acc_j = sum_i (x_q[i] - x_zp)*(w_q[j,i] - w_zp) + b_q[j]
    relu_acc = max(acc_j, 0)
    """
    x_int = x_q.to(torch.int32) - int(x_zp)
    w_int = w_q.to(torch.int32) - int(w_zp)

    acc = torch.matmul(w_int, x_int) + b_q
    relu_acc = torch.clamp(acc, min=0)
    return acc.to(torch.int32), relu_acc.to(torch.int32)

# HEX 导出

In [10]:
def write_int8_hex_1d(tensor, path):
    with open(path, "w") as f:
        for v in tensor.flatten():
            f.write(f"{int(v.item()) & 0xFF:02x}\n")

def write_int32_hex_1d(tensor, path):
    with open(path, "w") as f:
        for v in tensor.flatten():
            f.write(f"{int(v.item()) & 0xFFFFFFFF:08x}\n")

def export_route_a_artifacts(num_samples=10):
    # 加载训练后的模型
    model = MLPRouteA().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    # 只提取 fc1
    fc1_w_fp = model.fc1.weight.data.cpu()   # [128, 784]
    fc1_b_fp = model.fc1.bias.data.cpu()     # [128]

    # 固定输入量化参数
    input_scale = 1.0 / 255.0
    input_zero_point = -128

    # 权重量化（对称 int8）
    fc1_w_q, fc1_w_scale, fc1_w_zp = quantize_weight_symmetric_int8(fc1_w_fp)

    # 偏置量化（int32）
    fc1_b_q, fc1_b_scale = quantize_bias_int32(fc1_b_fp, input_scale, fc1_w_scale)

    # 导出权重：按 [out][in] 顺序
    write_int8_hex_1d(fc1_w_q.reshape(-1), os.path.join(OUT_DIR, "fc1_weight_int8.hex"))
    write_int32_hex_1d(fc1_b_q, os.path.join(OUT_DIR, "fc1_bias_int32.hex"))

    # 保存配置
    config = {
        "layer": "fc1_only",
        "weight_shape": [128, 784],
        "weight_layout": "row-major [out][in]",
        "input_layout": "[in]",
        "bias_layout": "[out]",
        "input_scale": input_scale,
        "input_zero_point": input_zero_point,
        "fc1_weight_scale": fc1_w_scale,
        "fc1_weight_zero_point": fc1_w_zp,
        "fc1_bias_scale": fc1_b_scale,
        "fc1_bias_dtype": "int32"
    }

    with open(os.path.join(OUT_DIR, "quant_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    # 测试集：为了导出原始像素，不能 Normalize
    test_transform = transforms.Compose([
        transforms.ToTensor()
    ])
    test_dataset = datasets.MNIST(DATA_DIR, train=False, download=True, transform=test_transform)

    labels = []

    for i in range(num_samples):
        img_fp, label = test_dataset[i]  # [1,28,28], range [0,1]
        labels.append(int(label))

        img_flat = img_fp.view(-1).cpu()
        x_q, _, _ = quantize_input_hw_style(img_flat)

        acc_int32, relu_int32 = fc1_integer_reference(
            x_q, fc1_w_q, fc1_b_q, x_zp=input_zero_point, w_zp=fc1_w_zp
        )

        write_int8_hex_1d(x_q, os.path.join(OUT_DIR, f"input_{i}.hex"))
        write_int32_hex_1d(acc_int32, os.path.join(OUT_DIR, f"fc1_acc_golden_{i}.hex"))
        write_int32_hex_1d(relu_int32, os.path.join(OUT_DIR, f"fc1_relu_golden_{i}.hex"))

        print(f"Exported sample {i}, label={label}")

    with open(os.path.join(OUT_DIR, "labels.txt"), "w") as f:
        for lb in labels:
            f.write(f"{lb}\n")

    print("Route A export done.")

# 运行

In [11]:

if __name__ == "__main__":
    train_model(num_epochs=5, lr=1e-3)
    export_route_a_artifacts(num_samples=10)

[Epoch 1/5] loss=0.302582, test_acc=0.9464
[Epoch 2/5] loss=0.135395, test_acc=0.9655
[Epoch 3/5] loss=0.093423, test_acc=0.9689
[Epoch 4/5] loss=0.071417, test_acc=0.9727
[Epoch 5/5] loss=0.054696, test_acc=0.9757
Model saved to: ./route_a_output/mlp_route_a.pth
Exported sample 0, label=7
Exported sample 1, label=2
Exported sample 2, label=1
Exported sample 3, label=0
Exported sample 4, label=4
Exported sample 5, label=1
Exported sample 6, label=4
Exported sample 7, label=9
Exported sample 8, label=5
Exported sample 9, label=9
Route A export done.
